# Raccolta Dati da Reddit — r/Italia, keyword: *estate*
Raccolta tramite **Arctic Shift** (archivio pubblico Reddit per ricerca accademica).
Nessuna credenziale richiesta.

**Strategia**: cerchiamo direttamente i **commenti** che contengono la parola 'estate'
in r/Italia, insieme ai metadati del post padre. Questo è più efficiente che cercare post
e poi scaricare i loro commenti.

## 0. Installazione dipendenze

In [ ]:
# !pip install requests spacy tqdm
# !python -m spacy download it_core_news_sm

## 1. Configurazione

In [ ]:
import requests
import time
import pandas as pd
from datetime import datetime

SUBREDDIT       = "Italia"
KEYWORD         = "estate"
TARGET_COMMENTS = 700
OUTPUT_CSV      = f"corpus_{SUBREDDIT}_{KEYWORD}.csv"

BASE_URL = "https://arctic-shift.photon-reddit.com/api"
HEADERS  = {"User-Agent": "python:elnsm.progetto.estate:v1.0 (academic NLP project)"}

print("Configurazione pronta.")
print(f"  Subreddit : r/{SUBREDDIT}")
print(f"  Keyword   : '{KEYWORD}'")
print(f"  Obiettivo : {TARGET_COMMENTS} commenti")

## 2. Test connessione API

In [ ]:
# Verifica che l'API risponda correttamente prima di procedere
test_url = f"{BASE_URL}/comments/search"
test_params = {
    "subreddit": SUBREDDIT,
    "body"   : KEYWORD,
    "limit"    : 3,
    "after"    : "2022-01-01",
    "before"   : "2022-12-31"
}

resp = requests.get(test_url, headers=HEADERS, params=test_params, timeout=20)
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    data = resp.json()
    print(f"Risposta OK! Esempio commento:")
    if data.get("data"):
        print(data["data"][0].get("body", "")[:200])
    print(f"\nChiavi disponibili: {list(data['data'][0].keys()) if data.get('data') else 'nessuna'}")
else:
    print(f"Errore: {resp.text}")

## 3. Raccolta commenti con paginazione temporale

In [ ]:
def collect_comments(subreddit, keyword, target=700):
    """
    Raccoglie commenti da Arctic Shift usando finestre temporali per la paginazione.
    Arctic Shift richiede always un range temporale (after/before).
    Scorriamo anno per anno dal più recente al più vecchio.
    """
    all_comments = []

    # Finestre temporali: scorriamo dal 2023 al 2018
    time_windows = [
        ("2023-01-01", "2024-01-01"),
        ("2022-01-01", "2023-01-01"),
        ("2021-01-01", "2022-01-01"),
        ("2020-01-01", "2021-01-01"),
        ("2019-01-01", "2020-01-01"),
        ("2018-01-01", "2019-01-01"),
    ]

    for after, before in time_windows:
        if len(all_comments) >= target:
            break

        print(f"\nFinestra {after[:4]}: raccolta commenti...")

        params = {
            "subreddit": subreddit,
            "body"   : keyword,
            "limit"    : 100,
            "after"    : after,
            "before"   : before,
            "sort"     : "desc"
        }

        window_comments = []
        last_utc = None  # per la paginazione dentro la finestra

        while True:
            if last_utc:
                params["before"] = last_utc

            try:
                resp = requests.get(
                    f"{BASE_URL}/comments/search",
                    headers=HEADERS,
                    params=params,
                    timeout=20
                )
                resp.raise_for_status()
                data = resp.json()
            except Exception as e:
                print(f"  Errore: {e}")
                break

            items = data.get("data", [])
            if not items:
                break

            for c in items:
                body = c.get("body", "").strip()
                if body in ("", "[deleted]", "[removed]") or len(body) < 15:
                    continue

                ts = float(c.get("created_utc", 0))
                window_comments.append({
                    "post_id"          : c.get("link_id", "").replace("t3_", ""),
                    "post_title"       : c.get("link_title", ""),  # disponibile in alcuni record
                    "comment_id"       : c.get("id", ""),
                    "comment_text"     : body,
                    "comment_author"   : c.get("author", ""),
                    "comment_timestamp": datetime.utcfromtimestamp(ts).isoformat(),
                    "comment_score"    : c.get("score", 0),
                    "subreddit"        : c.get("subreddit", subreddit),
                    "permalink"        : "https://reddit.com" + c.get("permalink", "")
                                         if c.get("permalink") else ""
                })

            # Paginazione: prendi il timestamp del commento più vecchio
            oldest_utc = min(float(c.get("created_utc", 0)) for c in items)
            last_utc = datetime.utcfromtimestamp(oldest_utc).strftime("%Y-%m-%dT%H:%M:%S")

            print(f"  Trovati {len(window_comments)} commenti in {after[:4]}...", end="\r")

            if len(items) < 100:
                break  # ultima pagina della finestra

            time.sleep(0.8)

        all_comments.extend(window_comments)
        print(f"  Anno {after[:4]}: {len(window_comments)} commenti | Totale: {len(all_comments)}")
        time.sleep(1)

    return all_comments


print("Funzione definita. Avvio raccolta...")
raw_comments = collect_comments(SUBREDDIT, KEYWORD, target=TARGET_COMMENTS)
print(f"\n=== Raccolta completata: {len(raw_comments)} commenti grezzo ===")

## 4. Arricchimento con titoli post (opzionale)

In [ ]:
# Se link_title non è disponibile nei commenti, recuperiamo i titoli dei post
# cercando i post per ID tramite Arctic Shift

df_raw = pd.DataFrame(raw_comments)
missing_titles = df_raw[df_raw["post_title"] == ""]["post_id"].unique()
print(f"Post con titolo mancante: {len(missing_titles)}")

if len(missing_titles) > 0 and len(missing_titles) <= 200:
    # Recuperiamo i titoli a batch di 50
    title_map = {}
    batch_size = 50
    for i in range(0, len(missing_titles), batch_size):
        batch = missing_titles[i:i+batch_size]
        ids_str = ",".join(batch)
        try:
            resp = requests.get(
                f"{BASE_URL}/posts/ids",
                headers=HEADERS,
                params={"ids": ids_str},
                timeout=20
            )
            if resp.status_code == 200:
                for post in resp.json().get("data", []):
                    title_map[post["id"]] = post.get("title", "")
        except Exception as e:
            print(f"  Errore recupero titoli: {e}")
        time.sleep(0.5)
    
    # Aggiorniamo i titoli mancanti
    df_raw["post_title"] = df_raw.apply(
        lambda r: title_map.get(r["post_id"], r["post_title"]) if r["post_title"] == "" else r["post_title"],
        axis=1
    )
    print(f"Titoli recuperati: {len(title_map)}")
else:
    print("Tutti i titoli già presenti (o troppi post da recuperare).")

df_raw

## 5. Salvataggio CSV

In [ ]:
df_corpus = df_raw.drop_duplicates(subset="comment_id").reset_index(drop=True)
df_corpus.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Salvati {len(df_corpus)} commenti unici in '{OUTPUT_CSV}'")
print(f"Colonne: {list(df_corpus.columns)}")
df_corpus[["post_title", "comment_text", "comment_author", "comment_timestamp"]].head(5)

## 6. NLP: Tokenizzazione, Lemmatizzazione e POS-tagging
Modello italiano `it_core_news_sm` di spaCy.

In [ ]:
import spacy

nlp = spacy.load("it_core_news_sm")

def process_text(text):
    doc = nlp(str(text))
    return [
        {
            "token" : token.text,
            "lemma" : token.lemma_.lower(),
            "pos"   : token.pos_,
            "is_adj": token.pos_ == "ADJ"
        }
        for token in doc
        if not token.is_space and not token.is_punct
    ]

# Test su un commento di esempio
esempio = df_corpus["comment_text"].iloc[0]
print(f"Commento:\n{esempio}\n")
print("Analisi:")
for t in process_text(esempio):
    flag = " <- ADJ" if t["is_adj"] else ""
    print(f"  {t['token']:20s} | {t['lemma']:20s} | {t['pos']}{flag}")

In [ ]:
from tqdm.auto import tqdm

print(f"Processamento di {len(df_corpus)} commenti con spaCy...")

all_tokens = []
for _, row in tqdm(df_corpus.iterrows(), total=len(df_corpus)):
    for t in process_text(row["comment_text"]):
        all_tokens.append({"comment_id": row["comment_id"], **t})

df_tokens = pd.DataFrame(all_tokens)
df_adj    = df_tokens[df_tokens["is_adj"] == True]

print(f"\nToken totali     : {len(df_tokens)}")
print(f"Aggettivi trovati: {len(df_adj)}")
print(f"\nTop 20 aggettivi più frequenti:")
print(df_adj["lemma"].value_counts().head(20))

In [ ]:
tokens_file = f"tokens_{SUBREDDIT}_{KEYWORD}.csv"
df_tokens.to_csv(tokens_file, index=False, encoding="utf-8-sig")
print(f"Token salvati in '{tokens_file}'")

## 7. Statistiche del corpus

In [ ]:
import plotly.express as px

print("=" * 50)
print("STATISTICHE CORPUS")
print("=" * 50)
print(f"Commenti totali          : {len(df_corpus)}")
print(f"Post unici               : {df_corpus['post_id'].nunique()}")
print(f"Autori unici             : {df_corpus['comment_author'].nunique()}")
print(f"Token totali             : {len(df_tokens)}")
print(f"Aggettivi totali         : {len(df_adj)}")
print(f"Aggettivi unici (lemmi)  : {df_adj['lemma'].nunique()}")
lunghezze = df_corpus["comment_text"].str.split().str.len()
print(f"Lunghezza media commenti : {lunghezze.mean():.1f} parole")
print(f"Periodo temporale        : {df_corpus['comment_timestamp'].min()[:10]} → {df_corpus['comment_timestamp'].max()[:10]}")

# Distribuzione per anno
df_corpus["anno"] = df_corpus["comment_timestamp"].str[:4]
fig1 = px.bar(
    df_corpus["anno"].value_counts().sort_index().reset_index(),
    x="anno", y="count",
    title="Commenti per anno",
    labels={"anno": "Anno", "count": "Numero commenti"}
)
fig1.show()

# Distribuzione lunghezza commenti
fig2 = px.histogram(
    x=lunghezze, nbins=40,
    title="Distribuzione lunghezza commenti (parole)",
    labels={"x": "Parole", "y": "Commenti"}
)
fig2.show()

---
## Prossimo passo → `Fase3_EmotionDetection.ipynb`
Usa i file prodotti:
- `corpus_Italia_estate.csv` — commenti con metadati
- `tokens_Italia_estate.csv` — token con POS-tag

Per fare il matching dei **lemmi aggettivali** con la matrice ELIta originale e ricalcolata.